In [ ]:
# Fusion (Spatial + DCT)
import os, cv2, numpy as np, random
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

Mounted at /content/drive
Spatial: (10200, 128, 128, 1) DCT: (10200, 128, 128, 1) labels: (10200,)
Epoch 1/10
234/234 ━━━━━━━━━━━━━━━━━━━━ 228s 953ms/step - accuracy: 0.6056 - loss: 0.6660 - val_accuracy: 0.7387 - val_loss: 0.4935
Epoch 2/10
234/234 ━━━━━━━━━━━━━━━━━━━━ 221s 944ms/step - accuracy: 0.7925 - loss: 0.4398 - val_accuracy: 0.9877 - val_loss: 0.0521
Epoch 3/10
234/234 ━━━━━━━━━━━━━━━━━━━━ 261s 939ms/step - accuracy: 0.9605 - loss: 0.1032 - val_accuracy: 0.9735 - val_loss: 0.0709
Epoch 4/10
234/234 ━━━━━━━━━━━━━━━━━━━━ 219s 936ms/step - accuracy: 0.9833 - loss: 0.0510 - val_accuracy: 0.9931 - val_loss: 0.0285
Epoch 5/10
234/234 ━━━━━━━━━━━━━━━━━━━━ 266s 952ms/step - accuracy: 0.9899 - loss: 0.0387 - val_accuracy: 0.9961 - val_loss: 0.0172
Epoch 6/10
234/234 ━━━━━━━━━━━━━━━━━━━━ 224s 960ms/step - accuracy: 0.9880 - loss: 0.0363 - val_accuracy: 0.9828 - val_loss: 0.0503
Epoch 7/10
234/234 ━━━━━━━━━━━━━━━━━━━━ 216s 925ms/step - accuracy: 0.9920 - loss: 0.0252 - val_accuracy: 0.9

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SEED=15
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

In [ ]:
REAL_DIR = "/content/drive/My Drive/Research Project/ffhq_7k/train"
FAKE_DIR = "/content/drive/My Drive/Research Project/StyleGAN2_7k/train"
IMG_SIZE=(128,128)

In [ ]:
def load_spatial_and_dct(real_dir, fake_dir, img_size=(128,128), limit=None):
    Xs, Xf, y = [], [], []
    def proc_sp(p):
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, img_size).astype(np.float32)/255.0
        return img[..., None]
    def proc_dct(p):
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, img_size).astype(np.float32)/255.0
        freq = cv2.dct(img)
        f = np.log(np.abs(freq)+1e-8)
        return f[..., None]
    reals = sorted(os.listdir(real_dir))
    fakes = sorted(os.listdir(fake_dir))
    if limit: reals, fakes = reals[:limit], fakes[:limit]
    for fn in reals:
        p = os.path.join(real_dir, fn)
        Xs.append(proc_sp(p)); Xf.append(proc_dct(p)); y.append(0)
    for fn in fakes:
        p = os.path.join(fake_dir, fn)
        Xs.append(proc_sp(p)); Xf.append(proc_dct(p)); y.append(1)
    return np.array(Xs), np.array(Xf), np.array(y)

In [ ]:
X_sp, X_dct, y = load_spatial_and_dct(REAL_DIR, FAKE_DIR)
print("Spatial:", X_sp.shape, "DCT:", X_dct.shape, "labels:", y.shape)

In [ ]:
def build_branch(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv2D(16,(4,4),activation='relu')(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32,(3,3),activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32,(3,3),activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64,activation='relu')(x)
    return models.Model(inp, x)

In [ ]:
def build_fusion(input_shape):
    a = build_branch(input_shape)
    b = build_branch(input_shape)
    comb = layers.Concatenate()([a.output, b.output])
    x = layers.Dense(64, activation='relu')(comb)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model([a.input, b.input], out)

In [ ]:
model = build_fusion((128,128,1))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
Xs_tr, Xs_te, Xd_tr, Xd_te, y_tr, y_te = train_test_split(X_sp, X_dct, y, test_size=0.2, random_state=SEED, stratify=y)

early_stop = EarlyStopping(monitor='val_loss', patience=9, restore_best_weights=True)
history = model.fit([Xs_tr, Xd_tr], y_tr, validation_data=([Xs_te, Xd_te], y_te), epochs=10, batch_size=35, callbacks=[early_stop], verbose=1)

print("Test eval:", model.evaluate([Xs_te, Xd_te], y_te, verbose=0))

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

from google.colab import drive
drive.mount('/content/drive')

REAL_TEST_DIR = "/content/drive/My Drive/Research Project/ffhq_7k/test"
FAKE_TEST_DIR = "/content/drive/My Drive/Research Project/StyleGAN3_7k/test"

Xs_test, Xf_test, y_test = load_spatial_and_dct(REAL_TEST_DIR, FAKE_TEST_DIR, img_size=IMG_SIZE, limit=None)

print("StyleGAN3 Test Shapes:", Xs_test.shape, Xf_test.shape, y_test.shape)


test_loss, test_acc = model.evaluate([Xs_test, Xf_test], y_test, verbose=1)
print("StyleGAN3 Test Loss:", test_loss)
print("StyleGAN3 Test Accuracy:", test_acc)


y_pred_probs = model.predict([Xs_test, Xf_test])
y_pred = (y_pred_probs > 0.5).astype("int32").flatten()

print("\nClassification Report (StyleGAN3 Test Set):")
print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
StyleGAN3 Test Shapes: (4079, 128, 128, 1) (4079, 128, 128, 1) (4079,)
128/128 ━━━━━━━━━━━━━━━━━━━━ 25s 192ms/step - accuracy: 0.9904 - loss: 0.0382
StyleGAN3 Test Loss: 0.07758426666259766
StyleGAN3 Test Accuracy: 0.9801421761512756
128/128 ━━━━━━━━━━━━━━━━━━━━ 26s 206ms/step

Classification Report (StyleGAN3 Test Set):
              precision    recall  f1-score   support

        Real       0.97      1.00      0.98      2079
        Fake       1.00      0.96      0.98      2000

    accuracy                           0.98      4079
   macro avg       0.98      0.98      0.98      4079
weighted avg       0.98      0.98      0.98      4079

